# examine MIRA scores of NDEs

In [1]:
import numpy as np
import glob, os
from tqdm import tqdm

In [2]:
import torch
from mira_score import mira

In [3]:
from px2cosmo import fm as FM 
from px2cosmo import util as UT

In [4]:
import corner as DFM
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rcParams['text.usetex'] = True
mpl.rcParams['font.family'] = 'serif'
mpl.rcParams['axes.linewidth'] = 1.5
mpl.rcParams['axes.xmargin'] = 1
mpl.rcParams['xtick.labelsize'] = 'x-large'
mpl.rcParams['xtick.major.size'] = 5
mpl.rcParams['xtick.major.width'] = 1.5
mpl.rcParams['ytick.labelsize'] = 'x-large'
mpl.rcParams['ytick.major.size'] = 5
mpl.rcParams['ytick.major.width'] = 1.5
mpl.rcParams['legend.frameon'] = False

# NLE

In [10]:
def MIRA_likelihood(zbin, study_dir, Nmocks=1000, L=1000):
    # construct test data
    _test_omegas = np.array([
        np.random.uniform(-1.8, -1.5, size=Nmocks), 
        np.random.uniform(-1.8, -1.2, size=Nmocks),
        np.random.uniform(-0.4, 0, size=Nmocks), 
        np.random.uniform(-22., -18., size=Nmocks)
    ]).T
    
    test_omegas, test_Xsig = [], []
    for i in tqdm(range(Nmocks)): 
        # forward model 
        _mock = FM.forwardmodel(_test_omegas[i], name=zbin, phi_amp=6e-3)
    
        test_omegas.append(np.tile(_test_omegas[i], (_mock.shape[0],1)))
        test_Xsig.append(_mock)
        
    test_omegas = np.vstack(test_omegas)
    test_Xsig = np.vstack(test_Xsig)
    
    test_omegasig = torch.tensor(np.hstack([test_omegas, test_Xsig[:,-2:]]).astype(np.float32))
    test_X = torch.tensor(test_Xsig[:,:2].astype(np.float32))

    ichoose = np.random.choice(np.arange(test_omegasig.shape[0]), size=L, replace=False)

    fmodels, test_samples = [], []
    for fmodel in tqdm(glob.glob(os.path.join(study_dir, '*.pt'))): 
        _model = torch.load(fmodel, weights_only=False, map_location='cpu')
        _model._device = 'cpu'
        test_samples.append(_model.sample_batched((1000,), x=test_omegasig[ichoose], show_progress_bars=False))
        fmodels.append(fmodel)
        
    test_samples = torch.tensor(np.array(test_samples).astype(np.float32)).permute(0, 2, 1, 3)
    means, stds = mira(test_X[ichoose], test_samples, num_runs=100)

    return fmodels, means, stds

In [6]:
fmodels_z14, means_z14, stds_z14 = MIRA_likelihood('z14', '/Users/ch54662/data/px2cosmo/mock/ndes/q_X_omegasig_z14_batch50/', Nmocks=1000, L=1000)

Mira MC runs: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:06<00:00, 14.56it/s]


In [11]:
for i in np.argsort(np.abs(means_z14.cpu().numpy() - 2/3)/stds_z14.cpu().numpy()): 
    print(os.path.basename(fmodels_z14[i]), means_z14[i].item(), stds_z14[i].item())

q_X_omegasig_z14_batch50.4.pt 0.6666549444198608 0.007335790898650885
q_X_omegasig_z14_batch50.19.pt 0.6666465997695923 0.006430050358176231
q_X_omegasig_z14_batch50.3.pt 0.6667296886444092 0.007420511916279793
q_X_omegasig_z14_batch50.13.pt 0.666577935218811 0.00856080837547779
q_X_omegasig_z14_batch50.21.pt 0.6665763258934021 0.006692506372928619
q_X_omegasig_z14_batch50.9.pt 0.6667698621749878 0.007204539142549038
q_X_omegasig_z14_batch50.16.pt 0.6669764518737793 0.007013925351202488
q_X_omegasig_z14_batch50.25.pt 0.6670733690261841 0.007432890124619007
q_X_omegasig_z14_batch50.8.pt 0.667329728603363 0.007628781255334616
q_X_omegasig_z14_batch50.2.pt 0.6659467816352844 0.007311349734663963
q_X_omegasig_z14_batch50.1.pt 0.6658250689506531 0.007268359884619713
q_X_omegasig_z14_batch50.17.pt 0.6675474643707275 0.007409057579934597
q_X_omegasig_z14_batch50.5.pt 0.6676197052001953 0.007922709919512272
q_X_omegasig_z14_batch50.10.pt 0.6657062768936157 0.007213358301669359
q_X_omegasig_z14

In [12]:
fmodels_z11, means_z11, stds_z11 = MIRA_likelihood('z11', '/Users/ch54662/data/px2cosmo/mock/ndes/nde/q_X_omegasig_z11/', Nmocks=1000, L=1000)

Mira MC runs: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:05<00:00, 19.17it/s]


In [13]:
for i in np.argsort(np.abs(means_z11.cpu().numpy() - 2/3)/stds_z11.cpu().numpy()): 
    print(os.path.basename(fmodels_z11[i]), means_z11[i].item(), stds_z11[i].item())

q_X_omegasig_z11.16.pt 0.6647135019302368 0.007708641700446606
q_X_omegasig_z11.17.pt 0.6647963523864746 0.006852686870843172
q_X_omegasig_z11.9.pt 0.6643096804618835 0.007699958048760891
q_X_omegasig_z11.10.pt 0.6641709804534912 0.007817275822162628
q_X_omegasig_z11.5.pt 0.6643660068511963 0.007196029182523489
q_X_omegasig_z11.8.pt 0.6641954779624939 0.0072474065236747265
q_X_omegasig_z11.12.pt 0.6642155647277832 0.007130621466785669
q_X_omegasig_z11.4.pt 0.6639323234558105 0.007800575345754623
q_X_omegasig_z11.20.pt 0.6641679406166077 0.006805281154811382
q_X_omegasig_z11.21.pt 0.664263129234314 0.006222351919859648
q_X_omegasig_z11.19.pt 0.6641832590103149 0.0062310462817549706
q_X_omegasig_z11.18.pt 0.6642464995384216 0.005813351832330227
q_X_omegasig_z11.3.pt 0.6634361743927002 0.007524267770349979
q_X_omegasig_z11.15.pt 0.6636437177658081 0.006811132188886404
q_X_omegasig_z11.2.pt 0.6636682748794556 0.006365431472659111
q_X_omegasig_z11.6.pt 0.6626408100128174 0.00811643339693546

In [14]:
fmodels_z9, means_z9, stds_z9 = MIRA_likelihood('z9', '/Users/ch54662/data/px2cosmo/mock/ndes/q_X_omegasig_z9_down/', Nmocks=1000, L=1000)

Mira MC runs: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:08<00:00, 12.42it/s]


In [15]:
for i in np.argsort(np.abs(means_z9.cpu().numpy() - 2/3)/stds_z9.cpu().numpy()): 
    print(os.path.basename(fmodels_z9[i]), means_z9[i].item(), stds_z9[i].item())

q_X_omegasig_z9_down.14.pt 0.6666380167007446 0.007179408334195614
q_X_omegasig_z9_down.7.pt 0.6668397784233093 0.007277180906385183
q_X_omegasig_z9_down.5.pt 0.6674192547798157 0.007105047814548016
q_X_omegasig_z9_down.42.pt 0.6675005555152893 0.007329788990318775
q_X_omegasig_z9_down.30.pt 0.6676360964775085 0.007558396551758051
q_X_omegasig_z9_down.19.pt 0.6675993204116821 0.0068074665032327175
q_X_omegasig_z9_down.9.pt 0.6675674319267273 0.006048888433724642
q_X_omegasig_z9_down.21.pt 0.6679269671440125 0.006270161829888821
q_X_omegasig_z9_down.46.pt 0.6679990887641907 0.0063745491206645966
q_X_omegasig_z9_down.38.pt 0.6682217121124268 0.007388515863567591
q_X_omegasig_z9_down.6.pt 0.6683111786842346 0.0077381860464811325
q_X_omegasig_z9_down.17.pt 0.6685006618499756 0.0077950116246938705
q_X_omegasig_z9_down.34.pt 0.6683880686759949 0.0063747502863407135
q_X_omegasig_z9_down.40.pt 0.6686159372329712 0.006567135453224182
q_X_omegasig_z9_down.41.pt 0.6689742803573608 0.0075781424529

# Poisson NDE

In [20]:
def MIRA_poisson(zbin, study_dir, Nmocks=1000): 
    # construct test data
    test_omegas = np.array([
        np.random.uniform(-1.8, -1.5, size=Nmocks), 
        np.random.uniform(-1.8, -1.2, size=Nmocks),
        np.random.uniform(-0.4, 0, size=Nmocks), 
        np.random.uniform(-22., -18., size=Nmocks), 
        np.random.uniform(1, 10, size=Nmocks) * 1e-3
    ]).T
    
    test_N = []
    for i in tqdm(range(Nmocks)): 
        # forward model 
        _mock = FM.forwardmodel(test_omegas[i,:-1], name=zbin, phi_amp=test_omegas[i,-1])
        test_N.append(_mock.shape[0])
    
    _test_omega = torch.tensor(test_omegas.astype(np.float32))
    _test_log1pN = torch.tensor(np.log1p(np.array(test_N)).astype(np.float32))
    
    bounds = np.array([[-1.8, -1.5], [-1.8, -1.2], [-0.4, 0.], [-22., -18.], [1e-3, 1e-2]])

    L = _test_log1pN.shape[0]

    fmodels, test_samples = [], []
    for fmodel in tqdm(glob.glob(os.path.join(study_dir, '*.pt'))): 
        _model = torch.load(fmodel, weights_only=False, map_location='cpu')
        _model._device = 'cpu'
        _test_samples = _model.sample_batched((1000,), x=_test_log1pN[:,None], show_progress_bars=False)
        test_samples.append(UT.cdf_transform(_test_samples, bounds.T).numpy())
        fmodels.append(fmodel)
        
    test_samples = torch.tensor(np.array(test_samples).astype(np.float32)).permute(0, 2, 1, 3)
    means, stds = mira(_test_omega, test_samples, num_runs=100)

    return fmodels, means, stds

In [21]:
Nmocks = 1000
fmodels_z14, means_z14, stds_z14 = MIRA_poisson('z14', '/Users/ch54662/data/px2cosmo/mock/ndes/q_omegat_log1pN_z14/', Nmocks=Nmocks)

Mira MC runs: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:12<00:00,  7.72it/s]


In [22]:
for i in np.argsort(np.abs(means_z14.cpu().numpy() - 2/3)/stds_z14.cpu().numpy()): 
    print(os.path.basename(fmodels_z14[i]), means_z14[i].item(), stds_z14[i].item())

q_omegat_log1pN_z14.45.pt 0.6667625904083252 0.007865783758461475
q_omegat_log1pN_z14.18.pt 0.6665701270103455 0.007703704293817282
q_omegat_log1pN_z14.21.pt 0.6667872667312622 0.006360047496855259
q_omegat_log1pN_z14.41.pt 0.6664843559265137 0.0063576651737093925
q_omegat_log1pN_z14.49.pt 0.6664678454399109 0.006581480614840984
q_omegat_log1pN_z14.12.pt 0.6664024591445923 0.008394218981266022
q_omegat_log1pN_z14.2.pt 0.6662366986274719 0.007717083673924208
q_omegat_log1pN_z14.15.pt 0.6661978363990784 0.007266883738338947
q_omegat_log1pN_z14.33.pt 0.6662134528160095 0.006671803072094917
q_omegat_log1pN_z14.26.pt 0.6660175323486328 0.007208303082734346
q_omegat_log1pN_z14.44.pt 0.6658802628517151 0.007382876239717007
q_omegat_log1pN_z14.25.pt 0.6659060120582581 0.00649298308417201
q_omegat_log1pN_z14.22.pt 0.6658595204353333 0.006580416113138199
q_omegat_log1pN_z14.24.pt 0.6656789183616638 0.007495529018342495
q_omegat_log1pN_z14.4.pt 0.6656137704849243 0.007395863998681307
q_omegat_log

In [26]:
fmodels_z11, means_z11, stds_z11 = MIRA_poisson('z11', '/Users/ch54662/data/px2cosmo/mock/ndes/nde/q_omegat_log1pN_z11/', Nmocks=Nmocks)

Mira MC runs: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:13<00:00,  7.48it/s]


In [27]:
for i in np.argsort(np.abs(means_z11.cpu().numpy() - 2/3)/stds_z11.cpu().numpy()): 
    print(os.path.basename(fmodels_z11[i]), means_z11[i].item(), stds_z11[i].item())

q_omegat_log1pN_z11.29.pt 0.6666592359542847 0.007437817752361298
q_omegat_log1pN_z11.26.pt 0.6666238307952881 0.006586933042854071
q_omegat_log1pN_z11.21.pt 0.6664975881576538 0.006011142861098051
q_omegat_log1pN_z11.0.pt 0.6664317846298218 0.007163572125136852
q_omegat_log1pN_z11.50.pt 0.6669619083404541 0.007387696765363216
q_omegat_log1pN_z11.23.pt 0.6670396327972412 0.007433312479406595
q_omegat_log1pN_z11.18.pt 0.6670429706573486 0.007493090350180864
q_omegat_log1pN_z11.25.pt 0.6670369505882263 0.007098703645169735
q_omegat_log1pN_z11.47.pt 0.6671142578125 0.007273341994732618
q_omegat_log1pN_z11.19.pt 0.667152464389801 0.0071226502768695354
q_omegat_log1pN_z11.40.pt 0.6673890948295593 0.00751611078158021
q_omegat_log1pN_z11.22.pt 0.6675298810005188 0.00743666710332036
q_omegat_log1pN_z11.13.pt 0.6659537553787231 0.006101615261286497
q_omegat_log1pN_z11.33.pt 0.6676960587501526 0.007398678921163082
q_omegat_log1pN_z11.27.pt 0.6676852703094482 0.006948144640773535
q_omegat_log1pN_

In [28]:
fmodels_z9, means_z9, stds_z9 = MIRA_poisson('z9', '/Users/ch54662/data/px2cosmo/mock/ndes/nde/q_omegat_log1pN_z9/', Nmocks=Nmocks)

Mira MC runs: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:34<00:00,  2.91it/s]


In [29]:
for i in np.argsort(np.abs(means_z9.cpu().numpy() - 2/3)/stds_z9.cpu().numpy()): 
    print(os.path.basename(fmodels_z9[i]), means_z9[i].item(), stds_z9[i].item())

q_omegat_log1pN_z9.49.pt 0.6664796471595764 0.006942758336663246
q_omegat_log1pN_z9.17.pt 0.6664108037948608 0.006618596613407135
q_omegat_log1pN_z9.10.pt 0.6670683026313782 0.006526350509375334
q_omegat_log1pN_z9.51.pt 0.6670976877212524 0.0067450073547661304
q_omegat_log1pN_z9.42.pt 0.6671518087387085 0.007584178354591131
q_omegat_log1pN_z9.0.pt 0.6671290397644043 0.006523418240249157
q_omegat_log1pN_z9.8.pt 0.6660845875740051 0.0072422088123857975
q_omegat_log1pN_z9.39.pt 0.6673234701156616 0.007707917597144842
q_omegat_log1pN_z9.73.pt 0.6674090623855591 0.007680675946176052
q_omegat_log1pN_z9.15.pt 0.6676629781723022 0.0071095870807766914
q_omegat_log1pN_z9.59.pt 0.6677259206771851 0.0068772779777646065
q_omegat_log1pN_z9.12.pt 0.6680349707603455 0.007634347304701805
q_omegat_log1pN_z9.68.pt 0.6680158972740173 0.007386200129985809
q_omegat_log1pN_z9.16.pt 0.6681867837905884 0.007638100069016218
q_omegat_log1pN_z9.44.pt 0.6681056022644043 0.007136060856282711
q_omegat_log1pN_z9.56.p